In [ ]:
# Chạy cell này trước. Nếu vừa upgrade openai lần đầu, restart kernel rồi chạy lại từ đầu.
%pip install --upgrade openai python-dotenv brotli


In [ ]:
# Mục tiêu:
# Khi user hỏi thì gộp câu hỏi của user + dữ liệu trong file json thành 1 prompt.
# Sau đó gửi prompt đó vào OpenAI API.

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

BASE_DIR = Path.cwd()
load_dotenv(BASE_DIR / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")

if not OPENAI_API_KEY or OPENAI_API_KEY == "your_api_key_here":
    raise ValueError("Chưa cấu hình OPENAI_API_KEY trong file .env.")

with open(BASE_DIR / "traffic_laws.json", "r", encoding="utf-8-sig") as file_object:
    data = json.load(file_object)

all_laws = []
for law in data:
    all_laws.append(
        f"Mã luật: {law['id']}\n"
        f"Nội dung vi phạm: {law['topic']}\n"
        f"Nội dung chi tiết: {law['content']}\n"
        f"Phương tiện: {law['vehicle_type']}\n"
        f"Mức phạt: {law['penalty']}\n"
    )

laws_context = "\n\n---\n\n".join(all_laws)

question = "Đi oto vượt đèn đỏ phạt bao tiền?"

final_question = f"""
DỮ LIỆU LUẬT GIAO THÔNG:

{laws_context}

CÂU HỎI CỦA NGƯỜI DÙNG:
{question}
""".strip()

SYSTEM_INSTRUCTIONS = """
Bạn là chatbot demo hỏi đáp Luật giao thông Việt Nam.

Quy tắc trả lời:
- Chỉ trả lời dựa trên dữ liệu luật được cung cấp.
- Không sử dụng kiến thức bên ngoài.
- Không tự suy đoán mức phạt.
- Nếu không tìm thấy thông tin, trả lời:
  "Tôi chưa tìm thấy thông tin phù hợp trong bộ dữ liệu."
- Nêu mã quy định và nguồn tham khảo nếu có dùng dữ liệu.
- Trả lời ngắn gọn, dễ hiểu.
- Nhắc rằng kết quả chỉ phục vụ học tập, không thay thế tư vấn pháp lý.
"""

client = OpenAI(
    api_key=OPENAI_API_KEY,
    timeout=60,
    default_headers={"Accept-Encoding": "gzip, deflate"},
)

res = client.responses.create(
    model=OPENAI_MODEL,
    instructions=SYSTEM_INSTRUCTIONS,
    input=final_question,
    max_output_tokens=700,
).output_text.strip()

print(res)


In [ ]:
print(res)
